# Explore WikiPathways with published queries

Load three [WikiPathways examples](https://github.com/wikipathways/sparql-examples/tree/3032382efd9c60c3ded84786b1bff099645f1e4b/examples/WikiPathways), run one and save the catalogue.

In [28]:
from rdfsolve.api import QueryCollection
from rdfsolve.sparql_helper import SparqlHelper

endpoint = "https://sparql.wikipathways.org/sparql"
base = "https://raw.githubusercontent.com/wikipathways/sparql-examples/3032382efd9c60c3ded84786b1bff099645f1e4b/examples/WikiPathways"
names = {"001": "dataset metadata", "006": "organisms", "009": "mouse pathways"}

## Load the SHACL catalogue

In [29]:
catalogue = QueryCollection.from_source({
    "name": "wikipathways",
    "endpoint": endpoint,
    "sparql_examples": {"shacl_dumps": [f"{base}/{number}.ttl" for number in names]},
})
for original, name in zip(list(catalogue.queries), names.values()):
    catalogue.rename(original, name)
list(catalogue.queries)

['dataset metadata', 'organisms', 'mouse pathways']

## Which organisms have pathways?

In [30]:
helper = SparqlHelper(endpoint, timeout=30)
helper.queries = catalogue
print(catalogue.queries["organisms"].query)

PREFIX wp: <http://vocabularies.wikipathways.org/wp#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?organism (str(?label) as ?name)
WHERE {
    ?concept wp:organism ?organism ;
      wp:organismName ?label .
}


In [31]:
organisms = helper.run_query("organisms")
organisms["results"]["bindings"][:8]

[{'organism': {'type': 'uri',
   'value': 'http://purl.obolibrary.org/obo/NCBITaxon_9913'},
  'name': {'type': 'literal', 'value': 'Bos taurus'}},
 {'organism': {'type': 'uri',
   'value': 'http://purl.obolibrary.org/obo/NCBITaxon_9615'},
  'name': {'type': 'literal', 'value': 'Canis familiaris'}},
 {'organism': {'type': 'uri',
   'value': 'http://purl.obolibrary.org/obo/NCBITaxon_10090'},
  'name': {'type': 'literal', 'value': 'Mus musculus'}},
 {'organism': {'type': 'uri',
   'value': 'http://purl.obolibrary.org/obo/NCBITaxon_10116'},
  'name': {'type': 'literal', 'value': 'Rattus norvegicus'}},
 {'organism': {'type': 'uri',
   'value': 'http://purl.obolibrary.org/obo/NCBITaxon_7955'},
  'name': {'type': 'literal', 'value': 'Danio rerio'}},
 {'organism': {'type': 'uri',
   'value': 'http://purl.obolibrary.org/obo/NCBITaxon_6239'},
  'name': {'type': 'literal', 'value': 'Caenorhabditis elegans'}},
 {'organism': {'type': 'uri',
   'value': 'http://purl.obolibrary.org/obo/NCBITaxon_9606

## Find five mouse pathways

In [32]:
query = catalogue.queries["mouse pathways"].query
catalogue.add("five mouse pathways", query + "\nLIMIT 5", endpoint=endpoint)
pathways = helper.run_query("five mouse pathways")
pathways["results"]["bindings"]

[{'wpIdentifier': {'type': 'uri',
   'value': 'https://identifiers.org/wikipathways/WP1'},
  'pathway': {'type': 'uri',
   'value': 'https://identifiers.org/wikipathways/WP1_r137182'},
  'page': {'type': 'uri',
   'value': 'http://www.wikipathways.org/instance/WP1_r137182'}},
 {'wpIdentifier': {'type': 'uri',
   'value': 'https://identifiers.org/wikipathways/WP10'},
  'pathway': {'type': 'uri',
   'value': 'https://identifiers.org/wikipathways/WP10_r139876'},
  'page': {'type': 'uri',
   'value': 'http://www.wikipathways.org/instance/WP10_r139876'}},
 {'wpIdentifier': {'type': 'uri',
   'value': 'https://identifiers.org/wikipathways/WP103'},
  'pathway': {'type': 'uri',
   'value': 'https://identifiers.org/wikipathways/WP103_r136920'},
  'page': {'type': 'uri',
   'value': 'http://www.wikipathways.org/instance/WP103_r136920'}},
 {'wpIdentifier': {'type': 'uri',
   'value': 'https://identifiers.org/wikipathways/WP108'},
  'pathway': {'type': 'uri',
   'value': 'https://identifiers.org/w

## What classes do these pathways have?

In [33]:
iris = [row["pathway"]["value"] for row in pathways["results"]["bindings"]]
helper.find_classes_for_iris(iris)

{'https://identifiers.org/wikipathways/WP103_r136920': ['http://www.w3.org/2004/02/skos/core#Collection',
  'http://vocabularies.wikipathways.org/wp#Pathway'],
 'https://identifiers.org/wikipathways/WP108_r142196': ['http://www.w3.org/2004/02/skos/core#Collection',
  'http://vocabularies.wikipathways.org/wp#Pathway'],
 'https://identifiers.org/wikipathways/WP113_r137233': ['http://www.w3.org/2004/02/skos/core#Collection',
  'http://vocabularies.wikipathways.org/wp#Pathway'],
 'https://identifiers.org/wikipathways/WP10_r139876': ['http://www.w3.org/2004/02/skos/core#Collection',
  'http://vocabularies.wikipathways.org/wp#Pathway'],
 'https://identifiers.org/wikipathways/WP1_r137182': ['http://www.w3.org/2004/02/skos/core#Collection',
  'http://vocabularies.wikipathways.org/wp#Pathway']}

## Save and reopen the catalogue

In [34]:
catalogue.to_turtle("wikipathways-queries.ttl")
reopened = QueryCollection()
reopened.load_shacl("wikipathways-queries.ttl")
list(reopened.queries)

['dataset metadata', 'organisms', 'mouse pathways', 'five mouse pathways']

## Review the requests

In [35]:
helper.history

[QueryRun(name='organisms', endpoint='https://sparql.wikipathways.org/sparql', started_at='2026-09-16T08:19:11.316468+00:00', elapsed_seconds=0.12040082400199026, success=True, error=''),
 QueryRun(name='five mouse pathways', endpoint='https://sparql.wikipathways.org/sparql', started_at='2026-09-16T08:19:11.503803+00:00', elapsed_seconds=0.8441382899763994, success=True, error='')]

In [39]:
helper.close()